# 16 — External Momentum Validation Against Kenneth French Mom

This notebook is a final implementation sanity check. It compares the project's realised momentum spreads with the established Kenneth French US Momentum Factor (Mom) over the common 2000–2025 period.

## Pre-specified interpretation

The two constructions are not expected to match in level because French Mom uses a 2x3 size × prior-return sort with NYSE breakpoints, while this project uses a screened investment universe and global quintiles. The validation criterion is therefore **economically meaningful co-movement**, not equality of mean return or beta equal to one.

The project spread indexed by formation month t is realised in t+1, so it is aligned to the French factor by **holding month**, not by formation-month label.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.portfolio import quintile_returns, quintile_return_panel
from src.external_validation import read_french_monthly_mom, align_project_with_french, validation_summary, correlation_summary, benchmark_regression


## 1. Rebuild the frozen project momentum series


In [ ]:
MONTHLY = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
FRENCH_MOM = ROOT / 'data/raw/F-F_Momentum_Factor.csv'

clean, _ = build_clean_monthly_panel(MONTHLY, CleaningConfig())
history = build_monthly_history_panel(MONTHLY)
mom = assign_signal_quintiles(attach_momentum_to_formation_universe(clean, history))
qret = quintile_returns(mom)
vw = quintile_return_panel(qret, return_col='VW_Return')
ew = quintile_return_panel(qret, return_col='EW_Return')

print('French Mom file exists:', FRENCH_MOM.exists())


## 2. Read and align Kenneth French Mom by realised holding month


In [ ]:
french = read_french_monthly_mom(FRENCH_MOM)
aligned = align_project_with_french(vw, ew, french)

print('French source range:', french['Month'].min(), 'to', french['Month'].max())
print('Common validation range:', aligned['Month'].min(), 'to', aligned['Month'].max())
print('Common months:', len(aligned))
aligned.head()


## 3. Summary statistics


In [ ]:
validation_summary(aligned)


## 4. Correlations — primary validation diagnostic


In [ ]:
correlation_summary(aligned)


## 5. Diagnostic benchmark regressions


In [ ]:
pd.DataFrame({
    'ProjectVW': benchmark_regression(aligned, 'ProjectVW', hac_lag=6),
    'ProjectEW': benchmark_regression(aligned, 'ProjectEW', hac_lag=6),
}).T


## Interpretation gate

- A positive, economically meaningful correlation with French Mom supports the implementation sanity check even if mean returns differ materially.
- A weak correlation would trigger a targeted re-audit of timing, breakpoints, and return construction before the paper is frozen.
- Do not change the project's own signal definition to make it look more like French Mom after observing this comparison.
